# 2. Feature Engineering
**Autores:** Eduardo Estefanía Ovejero, David Blasco Tena y Alejandro Jesús Castro García

Origen de los datos: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

## Objetivo
El objetivo de este notebook es realizar el proceso de **Feature Engineering** (Ingeniería de Características) para transformar los datos analizados previamente en un formato óptimo que el algoritmo de Machine Learning pueda procesar de manera eficiente.

Se pretende:
- **Codificar las variables categóricas:** Traducir las variables de texto al lenguaje matemático, aplicando One-Hot Encoding para las nominales y mapeo lógico para las ordinales.
- **Escalar las variables numéricas:** Estandarizar las magnitudes de variables muy dispares (como la Edad y el Salario Mensual) para garantizar el correcto funcionamiento del algoritmo de Descenso de Gradiente.
- **Crear nuevas características (Feature Creation):** Generar nuevas variables a partir de la combinación lógica de las existentes para resaltar patrones y aportar mayor poder predictivo al modelo.
- **Seleccionar características (Feature Selection):** Descartar variables redundantes o altamente correlacionadas (multicolinealidad) para reducir el ruido y simplificar el aprendizaje del modelo.

## 2.1. Importación de librerias y carga del Dataset Preprocesado

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings

pd.set_option('display.max_columns', None)

In [2]:
warnings.filterwarnings(action='ignore')

In [3]:
df = pd.read_csv('employee_attrition_01.csv')
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


## 2.2. Separación entre variables predictoras y variable objetivo

En esta fase se separa la variable objetivo (`Attrition`) del resto de variables predictoras.
Además, se transforma la variable objetivo a formato binario para poder trabajar posteriormente con modelos de clasificación supervisada.

In [4]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

print("\nDistribución original de la variable objetivo:")
print(y.value_counts())

Dimensiones de X: (1470, 30)
Dimensiones de y: (1470,)

Distribución original de la variable objetivo:
Attrition
No     1233
Yes     237
Name: count, dtype: int64


## 2.3. Codificación de la variable objetivo
La variable objetivo `Attrition` presenta valores categóricos (`yes` y `No`).
Para que pueda ser utilizada por el modelo, se transforma a una representación binaria:
- `Yes`: 1
- `No`: 0

In [5]:
y = y.map({"No": 0, "Yes": 1})

print("Distribución codificada de y:")
print(y.value_counts())
y.head()

Distribución codificada de y:
Attrition
0    1233
1     237
Name: count, dtype: int64


0    1
1    0
2    1
3    0
4    0
Name: Attrition, dtype: int64

## 2.4. Codificación de variables categóricas
Se aplica **One-Hot Encoding** a las variables categóricas nominales para transformarlas en variables numéricas utilizables por el modelo.

Se utiliza `drop_first=True` para evitar redundancia entre categorias y reducir posibles problemas de multicolinealidad.

In [6]:
num_cols = X.select_dtypes(exclude="object").columns.tolist()

X['OverTime'] = X['OverTime'].map({'No': 0, 'Yes': 1})
X['Gender'] = X['Gender'].map({'Female': 0, 'Male': 1})

cat_cols = X.select_dtypes(include="object").columns.tolist()
cat_cols.remove('BusinessTravel')

print("Variables categóricas nominales:")
print(cat_cols)

X = pd.get_dummies(X, columns=cat_cols, drop_first=True, dtype=int)

Variables categóricas nominales:
['Department', 'EducationField', 'JobRole', 'MaritalStatus']


Se aplica **Codificación Ordinal** a las variables categóricas ordinales para transformarlas en valores numéricos utilizables por el modelo. 

In [7]:
X['BusinessTravel'] = X['BusinessTravel'].map({
    'Non-Travel': 0, 
    'Travel_Rarely': 1, 
    'Travel_Frequently': 2})

print("Dimensiones tras la codificación:", X.shape)
X.head()

Dimensiones tras la codificación: (1470, 43)


,Age,BusinessTravel,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
0,41,1,1102,1,2,2,0,94,3,2,4,5993,19479,8,1,11,3,1,0,8,0,1,6,4,0,5,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1
1,49,2,279,8,1,3,1,61,2,2,2,5130,24907,1,0,23,4,4,1,10,3,3,10,7,1,7,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0
2,37,1,1373,2,2,4,1,92,2,1,3,2090,2396,6,1,15,3,2,0,7,3,3,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1
3,33,2,1392,3,4,4,0,56,3,1,3,2909,23159,1,1,11,3,3,0,8,3,3,8,7,3,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0
4,27,1,591,2,1,1,1,40,3,1,2,3468,16632,9,0,12,3,4,1,6,3,3,2,2,2,2,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0


## 2.5. Selección de características

Basándonos en el Análisis Exploratorio (el Mapa de Calor) hecho en `01_eda_exploration.ipynb`, vimos que había variables casi idénticas. Las más claras son:

- `JobLevel`: Está correlacionada al ~95% con MonthlyIncome (Salario Mensual). Como el salario tiene mucho más detalle numérico, nos quedamos con el salario y borramos el nivel.

- `YearsInCurrentRole` y `YearsWithCurrManager`: Ambas están altísimamente correlacionadas con YearsAtCompany (Años en la empresa). Es lógico: si llevas 10 años en la empresa, llevas tiempo con tu jefe y en tu rol. Nos quedamos solo con la antigüedad general en la empresa.

- `PercentSalaryHike`: Suele estar correlacionada al 100% con PerformanceRating (Nota de desempeño), porque a los que sacan mejor nota, les suben el sueldo el mismo porcentaje.

In [8]:
cols_a_borrar = [
    'JobLevel', # Redundante con MonthlyIncome
    'YearsInCurrentRole', # Redundante con YearsAtCompany
    'YearsWithCurrManager', # Redundante con YearsAtCompany
    'PercentSalaryHike' # Redundante con PerformanceRating
]

X = X.drop(columns=cols_a_borrar)

# Quitamos las columnas borradas de nuestra lista de numéricas
num_cols = [col for col in num_cols if col not in cols_a_borrar]

print("Dimensiones de X tras la selección de características:", X.shape)
X.head()

Dimensiones de X tras la selección de características: (1470, 39)


,Age,BusinessTravel,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsSinceLastPromotion,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
0,41,1,1102,1,2,2,0,94,3,4,5993,19479,8,1,3,1,0,8,0,1,6,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1
1,49,2,279,8,1,3,1,61,2,2,5130,24907,1,0,4,4,1,10,3,3,10,1,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0
2,37,1,1373,2,2,4,1,92,2,3,2090,2396,6,1,3,2,0,7,3,3,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1
3,33,2,1392,3,4,4,0,56,3,3,2909,23159,1,1,3,3,0,8,3,3,8,3,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0
4,27,1,591,2,1,1,1,40,3,2,3468,16632,9,0,3,4,1,6,3,3,2,2,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0


## 2.6. Separacion del dataset en entrenamiento y prueba
Una vez transformadas las variables, el datset se divide en dos subconjuntos:
- **Entrenamiento (80%)**
- **Prueba (20%)**

Se utiliza una partición estratificada (`stratify=y`) para conservar la proporción de clases en ambos subconjuntos, dado que la variable objetivo presenta desablanceo.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1176, 39)
X_test: (294, 39)
y_train: (1176,)
y_test: (294,)


## 2.7. Escalado de variables numéricas
Se aplica `StandardScaler` únicamente sobre las variables numéricas del conjunto de entrenamiento.
Posteriormente, la misma transformación se aplica al conjunto de prueba.

Este procedimiento evita la **fuga de informacion** (*data leakage*), ya que el conjunto de preuba no participa en el ajuste del escalador.

In [10]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

X_train_scaled.head()

,Age,BusinessTravel,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsSinceLastPromotion,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,JobRole_Human Resources,JobRole_Laboratory Technician,JobRole_Manager,JobRole_Manufacturing Director,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single
1194,1.090194,1,1.049455,-0.899915,1.064209,-0.658710,0,-0.908436,1.795282,-0.647997,2.026752,0.931289,1.330763,0,-0.432065,0.240218,2.613100,2.261482,-0.605389,0.337621,-0.665706,-0.368024,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
128,-1.634828,1,-0.523449,-0.899915,-1.855332,0.260202,1,1.694111,0.373564,1.153526,-0.864408,0.682742,-1.083704,0,-0.432065,0.240218,0.247430,-1.072675,-0.605389,0.337621,-0.830071,-0.056884,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0
810,0.981193,1,-0.992080,-0.777610,-1.855332,-1.577622,1,-0.662913,0.373564,0.252765,2.347706,0.167705,0.123529,0,-0.432065,1.160403,0.247430,1.492061,0.190962,0.337621,0.813578,0.565398,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0
478,-1.307825,1,-0.453653,0.445433,-1.855332,-0.658710,1,-1.252169,0.373564,0.252765,-0.956202,1.667056,-0.681293,0,-0.432065,0.240218,-0.935405,-0.559727,-1.401740,0.337621,-0.008246,-0.679165,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0
491,0.654191,2,0.491086,-0.043784,2.037390,1.179114,1,0.319180,0.373564,0.252765,-0.185956,0.728362,0.123529,1,-0.432065,-0.679966,0.247430,-0.175017,0.190962,0.337621,0.156119,0.565398,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0


## 2.8. Exportación de los conjuntos preparados
Finalmente, se guardan los conjuntos de entrenamiento y prueba ya preparados para ser utilizados en la fase de modelado.

Se exportan por separado:
- Las variables predictoras de entrenamiento
- Las variables predictoras de prueba
- Las etiquetas de entrenamiento
- Las etiquetas de prueba

In [11]:
X_train_scaled.to_csv("X_train_escalado.csv", index=False)
X_test_scaled.to_csv("X_test_escalado.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Archivos exportados correctamente.")

Archivos exportados correctamente.
